## 앙상블 학습 (Ensemble Learning)

### 1. Voting (보팅)
- **서로 다른 알고리즘**의 학습기들을 결합해 예측
- **Hard Voting**: 다수결 원칙
- **Soft Voting**: 각 모델의 예측 확률을 합산 → 확률이 가장 큰 클래스로 예측

### 2. Bagging (배깅)
- **동일한 알고리즘**의 학습기를 여러 개 사용
- 데이터를 **부트스트래핑**(복원추출, 중복 허용)하여 각기 다른 학습 데이터로 학습
- 예: Random Forest

### 3. Boosting (부스팅)
- 여러 학습기를 **순차적으로** 학습
- 이전 학습기가 틀린 데이터에 **가중치**를 부여하여 다음 학습기가 집중 학습
- 예: AdaBoost, Gradient Boosting, LightGBM, XGBoost

### 4. Stacking (스태킹)
- 개별 학습기들의 **예측값을 다시 학습 데이터(meta-feature)**로 사용
- 이를 최종 메타 모델에 입력해 최종 예측 수행

---
| 구분 | 학습기 | 데이터 | 학습 방식 |
|---|---|---|---|
| Voting | 서로 다름 | 동일 | 병렬 |
| Bagging | 동일 | 부트스트래핑(중복 O) | 병렬 |
| Boosting | 동일 | 가중치 부여 | 순차 |
| Stacking | 서로 다름 | 예측값 재사용 | 2단계 |


### 5. 편향과 분산 트레이드오프
- 모델이 너무 간단하면 편향이 커지고, 모델이 너무 복잡하면 분산이 커짐
- 주로 배깅이나 부스팅에서는 결정트리를 위주로 씀, 결정트리의 과적합이 발생하는  편향 분산 트레이드 오프를 앙상블로 해결하곤함

In [1]:
import pandas as pd

from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score



cancer=load_breast_cancer()
df=pd.DataFrame(cancer.data,columns=cancer.feature_names)
df.head(3)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758


In [3]:
# 개별 모델
lr_clf=LogisticRegression(solver='liblinear')
knn_clf=KNeighborsClassifier(n_neighbors=8)

# 소프트보팅
vo_clf=VotingClassifier(estimators=[('LR',lr_clf),('KNN',knn_clf)],voting='soft')

X_train,X_test,y_train,y_test=train_test_split(cancer.data,cancer.target,test_size=0.2,random_state=42)

# 
vo_clf.fit(X_train,y_train)
pred=vo_clf.predict(X_test)
score=accuracy_score(y_test,pred)

score

0.9649122807017544

In [9]:
clf=[lr_clf,knn_clf,vo_clf]

for i in clf:
    i.fit(X_train,y_train)
    pred=i.predict(X_test)
    score=accuracy_score(y_test,pred)
    class_name=i.__class__.__name__
    print(f'{class_name} 정확도:{score}')

LogisticRegression 정확도:0.956140350877193
KNeighborsClassifier 정확도:0.956140350877193
VotingClassifier 정확도:0.9649122807017544
